# Region-level OCR on Kaggle CPU (third review C5/M5)

Cuts every PAGE-XML text region out of the 40 evaluation pages and reads it with Tesseract and EasyOCR
(`ocr/run_region_ocr.py`, see `context/decisions.md` D41-D42). Tesseract **5.3.4** (the main run's version)
is installed from conda-forge into its own environment. The script first re-reads the whole pages and
records whether they match the main run's OCR; it continues either way and saves those whole-page texts
as a same-engine baseline.

**Before running:**
1. Session options: **Accelerator = None (CPU)**, Internet = on. No GPU quota is used.
2. Add data: the private dataset with `ocr_eval40.jsonl`, and the dataset with the page images and
   PAGE-XML files (`reid2019_eval40.zip`, uploaded as a Kaggle dataset).
   To resume a stopped run, also add that run's output (it contains a `region/` folder).
3. Save Version, then "Save & Run All (Commit)". About 2-4 h; the browser can be closed.

When it finishes, download `region.zip` from the version's Output tab.


In [ ]:
import glob, json, os, shutil, subprocess, sys

REPO_DIR = "/tmp/bengali-postocr-sllm"
OUT_DIR = "/kaggle/working/region"
os.makedirs(OUT_DIR, exist_ok=True)
SETUP_LOG = f"{OUT_DIR}/setup.log"

def sh(cmd):
    """Run a shell command, append its output to setup.log, return True on success."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    with open(SETUP_LOG, "a") as f:
        f.write(f"\n$ {cmd}\nexit {r.returncode}\n{r.stdout[-3000:]}{r.stderr[-3000:]}")
    print(("ok   " if r.returncode == 0 else "FAIL ") + cmd)
    return r.returncode == 0

if not os.path.isdir(REPO_DIR):
    sh(f"git clone -q https://github.com/Ariffurrhmn/bengali-postocr-sllm.git {REPO_DIR}")
assert os.path.isdir(REPO_DIR), "git clone failed: is Internet on in Session options? See region/setup.log"
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
sh("pip install -q pytesseract easyocr==1.7.2 jiwer")

# Tesseract 5.3.4 from conda-forge, in its own prefix so Kaggle's Python is not touched.
sh("mkdir -p /tmp/mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /tmp/mm bin/micromamba")
sh("/tmp/mm/bin/micromamba create -y -q -p /tmp/tess -c conda-forge tesseract=5.3.4")
TESS = "/tmp/tess/bin/tesseract"
TESSDATA = f"{REPO_DIR}/.tessdata"
os.makedirs(TESSDATA, exist_ok=True)
for lang in ("ben", "eng", "osd"):
    sh(f"curl -sL -o {TESSDATA}/{lang}.traineddata https://github.com/tesseract-ocr/tessdata_best/raw/main/{lang}.traineddata")
if not os.path.exists(TESS):
    # Fallback so the night is not lost: the system Tesseract. The verify step records the version
    # mismatch and the whole-page re-OCR becomes the same-engine baseline.
    print("WARNING: conda-forge Tesseract 5.3.4 failed; using apt's tesseract (see region/setup.log)")
    sh("apt-get -qq update && apt-get -qq install -y tesseract-ocr")
    TESS = shutil.which("tesseract")
assert TESS, "no Tesseract at all; see region/setup.log"
print(subprocess.run([TESS, "--version"], capture_output=True, text=True).stdout.splitlines()[0])


In [ ]:
ocr_files = glob.glob("/kaggle/input/**/ocr_eval40.jsonl", recursive=True)
assert ocr_files, "Attach the dataset that contains ocr_eval40.jsonl"
OCR = ocr_files[0]
xmls = [p for p in glob.glob("/kaggle/input/**/*.xml", recursive=True)]
assert xmls, "Attach the dataset with the page images and PAGE-XML files"
DATASET_DIR = os.path.dirname(xmls[0])
print("OCR:", OCR, "| pages folder:", DATASET_DIR, len(os.listdir(DATASET_DIR)), "files")
# Resume: copy an earlier run's output, if attached.
for f in glob.glob("/kaggle/input/**/region/*", recursive=True):
    if not os.path.exists(f"{OUT_DIR}/{os.path.basename(f)}"):
        shutil.copy(f, OUT_DIR); print("resuming from", f)


In [ ]:
env = {**os.environ, "TESSDATA_PREFIX": TESSDATA, "TESSERACT_CMD": TESS}
with open(f"{OUT_DIR}/run.log", "a") as log:
    r = subprocess.run([sys.executable, "run_region_ocr.py", "--dataset-dir", DATASET_DIR, "--stored", OCR,
                        "--verify", "--min-verified", "0", "--out", f"{OUT_DIR}/ocr_region40.jsonl"],
                       cwd=f"{REPO_DIR}/ocr", env=env, stdout=log, stderr=subprocess.STDOUT)
print("exit", r.returncode)
print("".join(open(f"{OUT_DIR}/run.log").readlines()[-5:]))


In [ ]:
v = json.load(open(f"{OUT_DIR}/ocr_region40_verify.json"))
print("versions", v["versions"], "| identical to the main run: Tesseract", v["tesseract_identical"],
      "EasyOCR", v["easyocr_identical"], "of", v["pages"])
rows = [json.loads(l) for l in open(f"{OUT_DIR}/ocr_region40.jsonl", encoding="utf-8") if l.strip()]
print(len(rows), "pages,", sum(len(r["regions"]) for r in rows), "regions")
shutil.make_archive("/kaggle/working/region", "zip", "/kaggle/working", "region")
print("wrote /kaggle/working/region.zip")
